# Public OrPen Xmon — Route B Eigenmode
This direct public example lowers the layout-authored junction sheet at the
finite-metal mid-height, verifies its two structured PEC-owner contacts, and
prepares a manual LTlab Slurm handoff. It never submits or runs Palace.

## Design And Geometry Controls

In [ ]:
import json
from pathlib import Path

from orpen_sc_pdk.tech import OUTER_VACUUM_THICKNESS_UM

GEOMETRY_CONTROLS = {
    "route": "B",
    "component": "kosen2024_flip_chip_xmon_qubit",
    "port_name": "o_junction_lumped",
    "logical_metal": "D1_BOTTOM_M1",
    "inductance_h": 1e-12,
    "coupon_padding_um": 75.0,
    "air_below_thickness_um": float(OUTER_VACUUM_THICKNESS_UM),
    "air_above_thickness_um": float(OUTER_VACUUM_THICKNESS_UM),
}

## Meshing Controls

In [ ]:
MESH_CONTROLS = {"refined_mesh_size": 15.0, "max_mesh_size": 80.0}

## Solver Controls

In [ ]:
SOLVER_CONTROLS = {
    "num_modes": 2,
    "target": 5e9,
    "tolerance": 1e-6,
    "save": 0,
    "order": 1,
    "max_iterations": 400,
    "solver_type": "Default",
    "preconditioner": "Default",
    "device": "CPU",
}

## Execution Controls

In [ ]:
EXECUTION_CONTROLS = {
    "machine_profiles": ("ltlab-local", "ltlab-slurm", "f1-slurm"),
    "selected_profile": "ltlab-slurm",
    "executable": "palace",
    "setup_commands": ("module load palace",),
    "resources": {
        "nodes": 1,
        "ntasks": 1,
        "cpus_per_task": 1,
        "command_style": "binary",
    },
}

## Output And Run Identity Controls

In [ ]:
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = (
        Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
    )

OUTPUT_CONTROLS = {
    "run_dir": NOTEBOOK_DIR
    / ".artifacts"
    / "kosen2024_flip_chip_xmon_route_b_eigenmode",
    "output_formats": ("gds", "xao", "msh2", "json", "sbatch", "tar.gz"),
}

## Validation And Failure Controls

In [ ]:
VALIDATION_CONTROLS = {
    "msh_version": "2.2",
    "lumped_port_count": 1,
    "no_solver_run": True,
}

## Data Classification And Provenance

In [ ]:
PROVENANCE = {
    "classification": "public",
    "orpen_sc_pdk_revision": "a16e8a123ce3ebfbda30aba31024506c2dcfd0c8",
    "gsim_meshing_methodology": "8f5dc6c05255d003a9c6d8959537bcf8068379d3",
    "palace_runtime": "0.16.1",
    "palace_schema": "0.16.0",
    "route_b_port_contract": "exact overlap-bound terminal curves and finite PEC sidewalls",
}

In [ ]:
import gdsfactory as gf
import orpen_sc_pdk
from orpen_sc_pdk import LAYER, LAYER_STACK, get_material_records

from scgsim.palace import EigenmodeSim
from scgsim.sgb import build_kosen2024_flip_chip_xmon_stack

EPR_SPECS = {
    kind: {"thickness": 0.003, "permittivity": 10.0, "loss_tangent": 0.0}
    for kind in ("MA", "MS", "SA")
}

## Build Component

In [ ]:
orpen_sc_pdk.activate()
gf.clear_cache()
component = gf.get_component(GEOMETRY_CONTROLS["component"])
stack = build_kosen2024_flip_chip_xmon_stack(
    component=component,
    layer_stack=LAYER_STACK,
    material_records=get_material_records(),
    d0_top_ground_mask_layer=tuple(LAYER.D0_TOP_GROUND_MASK),
    indium_bump_layer=tuple(LAYER.D0_D1_INDIUM_BUMP),
    coupon_padding_um=GEOMETRY_CONTROLS["coupon_padding_um"],
    air_below_thickness_um=GEOMETRY_CONTROLS["air_below_thickness_um"],
    air_above_thickness_um=GEOMETRY_CONTROLS["air_above_thickness_um"],
)
solution_regions = stack["solution_regions"]
assert tuple(solution_regions) == (
    "AIR_BELOW",
    "D0_SUBSTRATE",
    "D0_TO_D1_GAP",
    "D1_SUBSTRATE",
    "AIR_ABOVE",
)
assert (
    len(
        {
            json.dumps(region["geometry"]["domain_bounds_um"], sort_keys=True)
            for region in solution_regions.values()
        }
    )
    == 1
)
assert (
    solution_regions["AIR_BELOW"]["geometry"]["z_max_um"]
    == solution_regions["D0_SUBSTRATE"]["geometry"]["z_min_um"]
)
assert (
    solution_regions["D1_SUBSTRATE"]["geometry"]["z_max_um"]
    == solution_regions["AIR_ABOVE"]["geometry"]["z_min_um"]
)
run_dir = OUTPUT_CONTROLS["run_dir"]
if run_dir.exists() and any(run_dir.iterdir()):
    raise FileExistsError(f"Preserving existing inspectable run folder: {run_dir}")

## Configure Problem And EPR

In [ ]:
sim = EigenmodeSim()
sim.set_geometry(component)
sim.set_stack(stack)
sim.set_output_dir(run_dir)
sim.set_surface_epr(representation=GEOMETRY_CONTROLS["route"], specs=EPR_SPECS)
sim.add_port(
    GEOMETRY_CONTROLS["port_name"],
    layer=GEOMETRY_CONTROLS["logical_metal"],
    layout_sheet=True,
    inductance=GEOMETRY_CONTROLS["inductance_h"],
)
sim.set_eigenmode(
    num_modes=SOLVER_CONTROLS["num_modes"],
    target=SOLVER_CONTROLS["target"],
    tolerance=SOLVER_CONTROLS["tolerance"],
    save=SOLVER_CONTROLS["save"],
)
sim.set_numerical(
    **MESH_CONTROLS,
    **{
        key: SOLVER_CONTROLS[key]
        for key in (
            "order",
            "tolerance",
            "max_iterations",
            "solver_type",
            "preconditioner",
            "device",
        )
    },
)

## Build Mesh

In [ ]:
mesh_path = sim.mesh()
assert f"$MeshFormat\n{VALIDATION_CONTROLS['msh_version']} 0 8" in mesh_path.read_text()
mesh_manifest = json.loads((run_dir / "metadata" / "mesh_manifest.json").read_text())
assert not [
    group
    for group in mesh_manifest["groups"]
    if group["section"] == "volumes"
    and group.get("physical_attribute", {}).get("material_kinds") == ["conductor"]
]
gate = json.loads(
    (
        run_dir
        / "metadata"
        / "semantic_geometry"
        / "engine_gate_gmsh_brep_conformality.json"
    ).read_text()
)
port_records = [
    record
    for record in gate["records"]
    if record["source_record_kind"] == "route_b_lumped_port_mesh"
]
assert len(port_records) == 2
assert all(
    record["status"] == "pass"
    and record["shared_mesh_node_count"] > 0
    and record["expected_terminal_curve_ids"] == record["credited_terminal_curve_ids"]
    and all(record["per_terminal_curve_shared_node_counts"].values())
    for record in port_records
)

## Write And Validate Config

In [ ]:
config_path = sim.write_config()
config = json.loads(config_path.read_text())
assert (
    len(config["Boundaries"]["LumpedPort"]) == VALIDATION_CONTROLS["lumped_port_count"]
)
port_config = config["Boundaries"]["LumpedPort"][0]
port_information = json.loads(
    (run_dir / "metadata" / "port_information.json").read_text()
)["ports"][0]
assert port_config["L"] == GEOMETRY_CONTROLS["inductance_h"]
assert port_config["Direction"] == port_information["direction"]

## Prepare And Inspect Handoff

In [ ]:
handoff = sim.prepare_handoff(
    profile=EXECUTION_CONTROLS["selected_profile"],
    executable=EXECUTION_CONTROLS["executable"],
    resources=EXECUTION_CONTROLS["resources"],
    setup_commands=EXECUTION_CONTROLS["setup_commands"],
)
assert (
    handoff.script_path.name == "run_palace.sbatch" and handoff.archive_path.is_file()
)

## Physics Analysis Results
No Palace process is run; no physics result is claimed.

In [ ]:
assert VALIDATION_CONTROLS["no_solver_run"]

## Simulation Performance / Benchmarks

In [ ]:
print(
    {
        "mesh": str(mesh_path),
        "config": str(config_path),
        "handoff": str(handoff.archive_path),
        "inspection_run_dir": str(run_dir),
    }
)